In [1]:
from __future__ import absolute_import
from __future__ import division
from __future__ import print_function

import sys, os
sys.path.append(os.path.abspath(os.path.join('../../', 'config')))

os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"]="1"

from environment.carla_9_4.env import CarlaEnv
from environment.carla_9_4.config import ConfigManager

import itertools
import numpy as np
import tensorflow as tf
import tensorflow.contrib.layers as layers
import time

import baselines.common.tf_util as U

# NOTE: not using baselines logger for now
# from baselines import logger
from baselines import deepq
from baselines.deepq.deepq import ActWrapper
from baselines.deepq.replay_buffer import ReplayBuffer
from baselines.deepq.utils import ObservationInput
from baselines.common.schedules import LinearSchedule

import vis_module

from gym import wrappers

from datetime import datetime

from models import CoRLModel
from atari_model import AtariModel

import matplotlib.pyplot as plt

import tensorboard_logging as tf_log

In [2]:
prefix = 'dqn_rgb_lr_5e4_g_95_straight_run1/'
ALTA_LOGS = '~/alta-logs/' + prefix
TB_LOGS_DIR = ALTA_LOGS+'tb/'
MODEL_SAVE_DIR = '~/saved_models/'
IMAGES_PATH = MODEL_SAVE_DIR+'alta-logs/images/' + prefix
VIDEO_PATH = MODEL_SAVE_DIR+'alta-logs/videos/' + prefix
FRAME_SKIP = 5

In [ ]:

config = ConfigManager(algo="DQN_semantic")

# env = CarlaEnv(config.config)
vis_wrapper = vis_module.vis(IMAGES_PATH, VIDEO_PATH, FRAME_SKIP)
logger = tf_log.Logger(TB_LOGS_DIR)
env = CarlaEnv(config=config.config, vis_wrapper=vis_wrapper, logger=logger, log_dir=ALTA_LOGS)
print('INPUT TYPE: ', env.config['input_type'])




Launching CARLA server...
/zfsauton2/home/audreyh/carla_0.9.6/CarlaUE4.sh
Attempting to start carla on GPU 3
Launched server at port: 24500
Waiting for server to finish setting up


In [30]:
def newAtariModel(inputs, num_actions, scope, reuse=False):
    with tf.variable_scope(scope, reuse=reuse):
        activation = tf.nn.relu
        convs1 = [
            [16, [3, 3], 1],
        ]
        convs2 = [
            [32, [3, 3], 1],
        ]
        net = inputs
        out_size, kernel, stride = convs1[0]
        net = tf.layers.conv2d(net, out_size, kernel, stride)
        out_size, kernel, stride = convs2[0]
        net = tf.layers.conv2d(net, out_size, kernel, stride)
        net = tf.squeeze(net)
        net = tf.reshape(net, [-1, 84, 84, 32])
        net = tf.layers.flatten(net)
        #--------
        net = tf.layers.dense(inputs=net, 
        units= 256, 
        activation=activation)
        net = tf.layers.dense(inputs=net, units=num_actions)
    return net

In [31]:
from gym.spaces import Box, Discrete
observation_space = Box(low=0,
    high=255,
    shape=(128, 128), dtype=np.float32)
action_space = Discrete(4)

In [32]:

with U.make_session():
    act, train, update_target, debug = deepq.build_train(
                make_obs_ph=lambda name: ObservationInput(observation_space, name=name),
                q_func=newAtariModel, #CoRLModel,
                num_actions=action_space.n,
                optimizer=tf.train.AdamOptimizer(learning_rate=5e-4),
                gamma=0.95,
                double_q=True
            )


ValueError: Variable deepq/eps already exists, disallowed. Did you mean to set reuse=True or reuse=tf.AUTO_REUSE in VarScope? Originally defined at:

  File "/zfsauton2/home/audreyh/git/baselines/baselines/deepq/build_graph.py", line 181, in build_act
    eps = tf.get_variable("eps", (), initializer=tf.constant_initializer(0))
  File "/zfsauton2/home/audreyh/git/baselines/baselines/deepq/build_graph.py", line 376, in build_train
    act_f = build_act(make_obs_ph, q_func, num_actions, scope=scope, reuse=reuse)
  File "<ipython-input-5-040029a7a964>", line 7, in <module>
    double_q=True


In [7]:
print(tf.__version__)

1.10.0


In [25]:
! pip uninstall 

NameError: name 'baselines' is not defined